# 📘 Notebook 9: Advanced OOP Patterns

---

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. Use **Composition** as an alternative to inheritance
2. Create **Mixins** for reusable behavior
3. Use Python's **`dataclasses`** for cleaner class definitions
4. Implement common **Design Patterns**: Singleton, Observer, Strategy, Factory
5. Understand **Composition vs Inheritance** tradeoffs

---

## 1. Composition over Inheritance

**Inheritance** models an "**is-a**" relationship:
- A `Dog` **is an** `Animal`
- A `Car` **is a** `Vehicle`

**Composition** models a "**has-a**" relationship:
- A `Car` **has an** `Engine`
- A `Computer` **has a** `CPU`, **has** `RAM`, **has a** `Disk`

### When to Prefer Composition

| Use Inheritance When... | Use Composition When... |
|------------------------|------------------------|
| There's a clear "is-a" relationship | There's a "has-a" relationship |
| You need polymorphism | You need flexibility to swap parts |
| The hierarchy is shallow (2-3 levels) | The hierarchy would be deep or complex |
| Subtypes share most behavior | Objects are assembled from parts |

> 💡 **Favor composition over inheritance** — it's more flexible and easier to maintain.

In [ ]:
# ❌ BAD: Deep inheritance creates rigid hierarchies
# class ElectricSelfDrivingSportsCar(ElectricCar, SelfDrivingCar, SportsCar):
#     pass  # Multiple inheritance nightmare!

# ✅ GOOD: Composition — assemble from parts

class Engine:
    def __init__(self, engine_type: str, horsepower: int):
        self.engine_type = engine_type
        self.horsepower = horsepower
    
    def start(self):
        return f"{self.engine_type} engine ({self.horsepower}HP) started! 🔧"
    
    def __repr__(self):
        return f"Engine('{self.engine_type}', {self.horsepower}HP)"


class Battery:
    def __init__(self, capacity_kwh: float):
        self.capacity_kwh = capacity_kwh
        self.charge_level = 100  # percent
    
    def charge(self):
        self.charge_level = 100
        return "🔋 Battery fully charged!"
    
    def __repr__(self):
        return f"Battery({self.capacity_kwh}kWh, {self.charge_level}%)"


class GPS:
    def __init__(self):
        self.destination = None
    
    def navigate(self, destination: str):
        self.destination = destination
        return f"📍 Navigating to {destination}..."


class Car:
    """A car COMPOSED of parts — not inherited from them."""
    
    def __init__(self, make: str, model: str, engine: Engine, 
                 battery: Battery = None, gps: GPS = None):
        self.make = make
        self.model = model
        self.engine = engine        # HAS-A engine
        self.battery = battery      # HAS-A battery (optional)
        self.gps = gps or GPS()     # HAS-A GPS
    
    def start(self):
        parts = [self.engine.start()]
        if self.battery:
            parts.append(f"Battery: {self.battery.charge_level}%")
        return f"{self.make} {self.model}: {' | '.join(parts)}"
    
    def __repr__(self):
        return f"Car('{self.make}', '{self.model}', {self.engine})"


# Build different cars by composing different parts!
sports_car = Car(
    "Ferrari", "F40",
    engine=Engine("V8 Twin-Turbo", 478)
)

electric_car = Car(
    "Tesla", "Model S",
    engine=Engine("Electric Motor", 670),
    battery=Battery(100)
)

print(sports_car.start())
print(electric_car.start())
print(electric_car.gps.navigate("San Francisco"))

# Easy to swap parts!
electric_car.engine = Engine("Upgraded Electric", 1020)
print(f"\nUpgraded: {electric_car.start()}")

---

## 2. Mixins

A **mixin** is a class that provides a specific piece of functionality meant to be "mixed in" to other classes. Mixins are NOT meant to stand alone.

### Rules for Mixins:
- Provide a **single, focused** piece of functionality
- Should NOT have `__init__` (usually)
- Name them with a `Mixin` suffix for clarity

In [ ]:
import json

# === MIXINS: Reusable behaviors ===

class SerializableMixin:
    """Adds JSON serialization capability."""
    
    def to_json(self):
        return json.dumps(self.__dict__, indent=2)
    
    @classmethod
    def from_json(cls, json_string):
        data = json.loads(json_string)
        return cls(**data)


class PrintableMixin:
    """Adds a pretty print method."""
    
    def pretty_print(self):
        class_name = self.__class__.__name__
        attrs = ", ".join(f"{k}={v!r}" for k, v in self.__dict__.items())
        print(f"┌─ {class_name} ─────────────")
        for k, v in self.__dict__.items():
            print(f"│  {k}: {v!r}")
        print(f"└{'─' * (len(class_name) + 15)}")


class ValidatableMixin:
    """Adds validation support."""
    _validators = {}
    
    def validate(self):
        errors = []
        for attr, validator in self._validators.items():
            value = getattr(self, attr, None)
            if not validator(value):
                errors.append(f"Invalid {attr}: {value!r}")
        return errors


# === USE MIXINS by mixing them in ===

class User(SerializableMixin, PrintableMixin):
    def __init__(self, name: str, email: str, age: int):
        self.name = name
        self.email = email
        self.age = age


class Product(SerializableMixin, PrintableMixin):
    def __init__(self, name: str, price: float, category: str):
        self.name = name
        self.price = price
        self.category = category


# Both User and Product now have serialization and pretty printing!
user = User("Alice", "alice@example.com", 30)
product = Product("Widget", 29.99, "Electronics")

user.pretty_print()
product.pretty_print()

# Serialize to JSON
json_str = user.to_json()
print(f"\nJSON:\n{json_str}")

# Deserialize from JSON
user2 = User.from_json(json_str)
print(f"\nDeserialized: {user2.name}, {user2.email}")

---

## 3. Dataclasses (`@dataclass`)

Python 3.7+ introduced **dataclasses** — a decorator that auto-generates boilerplate code like `__init__`, `__repr__`, `__eq__`, and more.

### Before vs After:

In [ ]:
# ❌ WITHOUT dataclass: lots of boilerplate
class PointOld:
    def __init__(self, x: float, y: float, z: float = 0):
        self.x = x
        self.y = y
        self.z = z
    
    def __repr__(self):
        return f"Point(x={self.x}, y={self.y}, z={self.z})"
    
    def __eq__(self, other):
        return (isinstance(other, PointOld) and 
                self.x == other.x and self.y == other.y and self.z == other.z)

print(f"Old style: {PointOld(1, 2)}")
print(f"Equal? {PointOld(1, 2) == PointOld(1, 2)}")

In [ ]:
# ✅ WITH dataclass: clean and concise!
from dataclasses import dataclass, field

@dataclass
class Point:
    x: float
    y: float
    z: float = 0  # Default value

# __init__, __repr__, __eq__ are auto-generated!
p1 = Point(1, 2)
p2 = Point(1, 2, 3)
p3 = Point(1, 2)

print(f"p1 = {p1}")             # Auto __repr__
print(f"p2 = {p2}")
print(f"p1 == p3: {p1 == p3}")  # Auto __eq__
print(f"p1 == p2: {p1 == p2}")

In [ ]:
# Advanced dataclass features
from dataclasses import dataclass, field
from typing import List

@dataclass(order=True)  # Auto-generates comparison methods!
class Student:
    # sort_index is used for comparison but hidden from repr
    sort_index: float = field(init=False, repr=False)
    name: str
    gpa: float
    courses: List[str] = field(default_factory=list)  # Mutable default!
    
    def __post_init__(self):
        """Called after __init__ — good for computed fields."""
        self.sort_index = self.gpa  # Sort by GPA


s1 = Student("Alice", 3.9, ["Math", "CS"])
s2 = Student("Bob", 3.5)
s3 = Student("Charlie", 4.0, ["Physics"])

print(s1)
print(s2)

# Auto-sorting works!
students = [s1, s2, s3]
print(f"\nSorted: {sorted(students)}")

# Each student has their OWN courses list (default_factory)
s2.courses.append("Art")
print(f"\nBob's courses: {s2.courses}")
print(f"Alice's courses: {s1.courses}")  # Unaffected!

In [ ]:
# Frozen (immutable) dataclasses
from dataclasses import dataclass

@dataclass(frozen=True)  # Makes instances immutable!
class Color:
    r: int
    g: int
    b: int
    
    @property
    def hex(self):
        return f"#{self.r:02x}{self.g:02x}{self.b:02x}"


red = Color(255, 0, 0)
print(f"{red} → {red.hex}")

# Can't modify!
try:
    red.r = 200
except Exception as e:
    print(f"❌ {type(e).__name__}: {e}")

# Frozen dataclasses are hashable — can use in sets!
colors = {Color(255, 0, 0), Color(0, 255, 0), Color(255, 0, 0)}
print(f"Unique colors: {len(colors)}")  # 2 (duplicate removed)

---

## 4. Design Patterns

Design patterns are **proven solutions** to common software design problems.

### 4.1 Singleton Pattern

Ensures only **one instance** of a class ever exists:

In [ ]:
class DatabaseConnection:
    """Singleton: only one database connection should exist."""
    _instance = None
    
    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            print("Creating new database connection...")
            cls._instance = super().__new__(cls)
        else:
            print("Returning existing connection.")
        return cls._instance
    
    def __init__(self, host: str = "localhost"):
        self.host = host
    
    def query(self, sql: str):
        return f"Executing on {self.host}: {sql}"


db1 = DatabaseConnection("server1")
db2 = DatabaseConnection("server2")  # Returns the SAME instance!

print(f"\ndb1 is db2: {db1 is db2}")  # True — same object!
print(f"db1.host: {db1.host}")  # Both point to same instance

### 4.2 Observer Pattern

Objects (**observers**) subscribe to events from another object (**subject**), and get notified when something changes:

In [ ]:
from abc import ABC, abstractmethod

class EventSystem:
    """A subject that observers can subscribe to."""
    
    def __init__(self):
        self._listeners = {}  # event_name → [callbacks]
    
    def on(self, event: str, callback):
        """Subscribe to an event."""
        if event not in self._listeners:
            self._listeners[event] = []
        self._listeners[event].append(callback)
    
    def emit(self, event: str, data=None):
        """Emit an event, notifying all listeners."""
        for callback in self._listeners.get(event, []):
            callback(data)


class Store(EventSystem):
    """An e-commerce store with event notifications."""
    
    def __init__(self):
        super().__init__()
        self.inventory = {}
    
    def add_product(self, name: str, price: float):
        self.inventory[name] = price
        self.emit("product_added", {"name": name, "price": price})
    
    def process_sale(self, product_name: str):
        if product_name in self.inventory:
            price = self.inventory[product_name]
            self.emit("sale", {"product": product_name, "price": price})


# Create the store
store = Store()

# Subscribe observers
store.on("product_added", lambda d: print(f"📦 New product: {d['name']} at ${d['price']}"))
store.on("sale", lambda d: print(f"💰 Sale! {d['product']} for ${d['price']}"))
store.on("sale", lambda d: print(f"📧 Email receipt for {d['product']}..."))
store.on("sale", lambda d: print(f"📊 Analytics: tracked sale of {d['product']}"))

# Actions trigger notifications
store.add_product("Widget", 29.99)
print()
store.process_sale("Widget")

### 4.3 Strategy Pattern

Define a **family of algorithms** and make them interchangeable:

In [ ]:
from abc import ABC, abstractmethod

# Strategy interface
class ShippingStrategy(ABC):
    @abstractmethod
    def calculate(self, weight: float) -> float:
        pass

# Concrete strategies
class StandardShipping(ShippingStrategy):
    def calculate(self, weight: float) -> float:
        return weight * 1.5  # $1.50 per pound

class ExpressShipping(ShippingStrategy):
    def calculate(self, weight: float) -> float:
        return weight * 3.0 + 5.0  # $3 per pound + $5 flat

class FreeShipping(ShippingStrategy):
    def calculate(self, weight: float) -> float:
        return 0


# Context: uses a strategy
class Order:
    def __init__(self, items: list, weight: float):
        self.items = items
        self.weight = weight
        self.shipping_strategy = StandardShipping()  # Default
    
    def set_shipping(self, strategy: ShippingStrategy):
        """Swap shipping strategy at runtime!"""
        self.shipping_strategy = strategy
    
    def get_shipping_cost(self) -> float:
        return self.shipping_strategy.calculate(self.weight)
    
    def get_total(self) -> float:
        item_total = sum(price for _, price in self.items)
        return item_total + self.get_shipping_cost()


# Usage
order = Order([("Widget", 10), ("Gadget", 25)], weight=5)

order.set_shipping(StandardShipping())
print(f"Standard: ${order.get_shipping_cost():.2f} shipping, ${order.get_total():.2f} total")

order.set_shipping(ExpressShipping())
print(f"Express:  ${order.get_shipping_cost():.2f} shipping, ${order.get_total():.2f} total")

order.set_shipping(FreeShipping())
print(f"Free:     ${order.get_shipping_cost():.2f} shipping, ${order.get_total():.2f} total")

### 4.4 Factory Pattern

A **factory** creates objects without exposing the creation logic:

In [ ]:
from abc import ABC, abstractmethod

class Notification(ABC):
    @abstractmethod
    def send(self, message: str):
        pass

class EmailNotification(Notification):
    def send(self, message):
        print(f"📧 Email: {message}")

class SMSNotification(Notification):
    def send(self, message):
        print(f"📱 SMS: {message}")

class PushNotification(Notification):
    def send(self, message):
        print(f"🔔 Push: {message}")


class NotificationFactory:
    """Factory: creates the right notification type based on a string."""
    
    _types = {
        "email": EmailNotification,
        "sms": SMSNotification,
        "push": PushNotification,
    }
    
    @classmethod
    def create(cls, notification_type: str) -> Notification:
        notification_class = cls._types.get(notification_type.lower())
        if notification_class is None:
            raise ValueError(f"Unknown type: {notification_type}")
        return notification_class()
    
    @classmethod
    def register(cls, name: str, notification_class):
        """Register a new notification type."""
        cls._types[name] = notification_class


# Usage — the caller doesn't need to know the classes
for ntype in ["email", "sms", "push"]:
    notification = NotificationFactory.create(ntype)
    notification.send("Your order has shipped!")

# Extensible — register new types without modifying existing code
class SlackNotification(Notification):
    def send(self, message):
        print(f"💬 Slack: {message}")

NotificationFactory.register("slack", SlackNotification)
NotificationFactory.create("slack").send("New deployment!")

---

## 5. Putting It All Together: A Mini Project

Let's build a small **Task Management System** using multiple OOP patterns:

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import List, Optional


# === Enums for type safety ===
class Priority(Enum):
    LOW = 1
    MEDIUM = 2
    HIGH = 3
    CRITICAL = 4

class Status(Enum):
    TODO = "todo"
    IN_PROGRESS = "in_progress"
    DONE = "done"


# === Dataclass for clean data modeling ===
@dataclass
class Task:
    title: str
    priority: Priority = Priority.MEDIUM
    status: Status = Status.TODO
    tags: List[str] = field(default_factory=list)
    created_at: datetime = field(default_factory=datetime.now)
    
    def complete(self):
        self.status = Status.DONE
    
    def start(self):
        self.status = Status.IN_PROGRESS


# === Composition: TaskBoard contains Tasks ===
class TaskBoard:
    def __init__(self, name: str):
        self.name = name
        self._tasks: List[Task] = []
        self._listeners = []  # Observer pattern
    
    # Observer pattern
    def add_listener(self, callback):
        self._listeners.append(callback)
    
    def _notify(self, event: str, task: Task):
        for callback in self._listeners:
            callback(event, task)
    
    # Core operations
    def add_task(self, task: Task):
        self._tasks.append(task)
        self._notify("added", task)
    
    def complete_task(self, title: str):
        for task in self._tasks:
            if task.title == title:
                task.complete()
                self._notify("completed", task)
                return
        raise ValueError(f"Task '{title}' not found")
    
    # Container methods (magic methods)
    def __len__(self):
        return len(self._tasks)
    
    def __iter__(self):
        return iter(sorted(self._tasks, key=lambda t: t.priority.value, reverse=True))
    
    def __contains__(self, title: str):
        return any(t.title == title for t in self._tasks)
    
    # Filtering (using properties for clean API)
    @property
    def pending(self):
        return [t for t in self._tasks if t.status != Status.DONE]
    
    @property
    def completed(self):
        return [t for t in self._tasks if t.status == Status.DONE]
    
    def summary(self):
        total = len(self._tasks)
        done = len(self.completed)
        print(f"\n📋 {self.name}")
        print(f"   Total: {total} | Done: {done} | Pending: {total - done}")
        print(f"   Progress: {'█' * done}{'░' * (total - done)} {done}/{total}")
        for task in self:
            icon = "✅" if task.status == Status.DONE else "🔲"
            priority_icon = {Priority.LOW: "🟢", Priority.MEDIUM: "🟡", 
                           Priority.HIGH: "🟠", Priority.CRITICAL: "🔴"}[task.priority]
            print(f"   {icon} {priority_icon} {task.title} [{task.status.value}]")


# === Build and use the system ===
board = TaskBoard("Sprint 42")

# Add observer
board.add_listener(lambda event, task: 
    print(f"📣 Event: task '{task.title}' {event}!"))

# Add tasks
board.add_task(Task("Fix login bug", Priority.CRITICAL, tags=["bug", "auth"]))
board.add_task(Task("Add dark mode", Priority.MEDIUM, tags=["feature", "ui"]))
board.add_task(Task("Update docs", Priority.LOW, tags=["docs"]))
board.add_task(Task("Optimize queries", Priority.HIGH, tags=["performance"]))

# Complete some tasks
board.complete_task("Fix login bug")
board.complete_task("Update docs")

# Summary
board.summary()

# Container operations
print(f"\n'Add dark mode' in board: {'Add dark mode' in board}")
print(f"Total tasks: {len(board)}")

---

## 🏋️ Practice Exercises

### Exercise 1: Plugin System
Create a plugin system using composition:
- An abstract `Plugin` class with `execute(data)` method
- Concrete plugins: `UpperCasePlugin`, `ReversePlugin`, `CensorPlugin`
- A `TextProcessor` class that accepts a list of plugins and applies them in order

In [ ]:
# Exercise 1: Your code here



### Exercise 2: Dataclass Practice
Create a `Library` system using dataclasses:
- `@dataclass Book` with title, author, isbn, available (bool)
- `@dataclass Member` with name, member_id, borrowed_books (list)
- A regular `Library` class that manages books and members with checkout/return methods

In [ ]:
# Exercise 2: Your code here



---

### ✅ Solutions

In [ ]:
# Solution 1: Plugin System
from abc import ABC, abstractmethod

class Plugin(ABC):
    @abstractmethod
    def execute(self, data: str) -> str:
        pass

class UpperCasePlugin(Plugin):
    def execute(self, data: str) -> str:
        return data.upper()

class ReversePlugin(Plugin):
    def execute(self, data: str) -> str:
        return data[::-1]

class CensorPlugin(Plugin):
    def __init__(self, bad_words: list):
        self.bad_words = bad_words
    
    def execute(self, data: str) -> str:
        for word in self.bad_words:
            data = data.replace(word, '*' * len(word))
        return data

class TextProcessor:
    def __init__(self):
        self.plugins: list[Plugin] = []
    
    def add_plugin(self, plugin: Plugin):
        self.plugins.append(plugin)
        return self  # Enable chaining
    
    def process(self, text: str) -> str:
        for plugin in self.plugins:
            text = plugin.execute(text)
        return text

# Usage
processor = TextProcessor()
processor.add_plugin(CensorPlugin(["bad", "ugly"])).add_plugin(UpperCasePlugin())

result = processor.process("This is a bad and ugly sentence.")
print(f"Processed: {result}")

In [ ]:
# Solution 2: Library with Dataclasses
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class Book:
    title: str
    author: str
    isbn: str
    available: bool = True

@dataclass
class Member:
    name: str
    member_id: str
    borrowed_books: List[str] = field(default_factory=list)

class Library:
    def __init__(self, name: str):
        self.name = name
        self._books: dict[str, Book] = {}    # isbn → Book
        self._members: dict[str, Member] = {}  # id → Member
    
    def add_book(self, book: Book):
        self._books[book.isbn] = book
    
    def register_member(self, member: Member):
        self._members[member.member_id] = member
    
    def checkout(self, isbn: str, member_id: str) -> bool:
        book = self._books.get(isbn)
        member = self._members.get(member_id)
        if not book or not member:
            print("❌ Book or member not found")
            return False
        if not book.available:
            print(f"❌ '{book.title}' is not available")
            return False
        book.available = False
        member.borrowed_books.append(isbn)
        print(f"✅ {member.name} borrowed '{book.title}'")
        return True
    
    def return_book(self, isbn: str, member_id: str):
        book = self._books.get(isbn)
        member = self._members.get(member_id)
        if book and member and isbn in member.borrowed_books:
            book.available = True
            member.borrowed_books.remove(isbn)
            print(f"✅ {member.name} returned '{book.title}'")

# Test
lib = Library("City Library")
lib.add_book(Book("Python Crash Course", "Eric Matthes", "978-1"))
lib.add_book(Book("Clean Code", "Robert Martin", "978-2"))
lib.register_member(Member("Alice", "M001"))

lib.checkout("978-1", "M001")
lib.checkout("978-1", "M001")  # Already checked out
lib.return_book("978-1", "M001")

---

## 📌 Key Takeaways

1. **Composition > Inheritance** for flexibility — "has-a" > "is-a" when in doubt
2. **Mixins** provide reusable behavior without deep inheritance hierarchies
3. **`@dataclass`** eliminates boilerplate for data-heavy classes
4. **Design Patterns** are proven solutions: Singleton, Observer, Strategy, Factory
5. Combine patterns thoughtfully — don't over-engineer simple problems

---

## 🎓 Congratulations!

You've completed the Python OOP series! Here's a summary of everything you've learned:

| Notebook | Topic | Key Concepts |
|----------|-------|--------------|
| 01 | Introduction to OOP | Classes, objects, attributes, methods, `self` |
| 02 | Constructors & `__init__` | Initialization, type hints, defaults, assertions |
| 03 | Class vs Instance Attributes | Shared vs unique data, `__dict__`, lookup chain |
| 04 | Class & Static Methods | `@classmethod`, `@staticmethod`, factory methods |
| 05 | Inheritance | `super()`, method overriding, MRO, multiple inheritance |
| 06 | Encapsulation | Properties, getters, setters, name mangling |
| 07 | Four Pillars | Abstraction (ABCs), polymorphism, duck typing |
| 08 | Magic Methods | Dunder methods, operator overloading, context managers |
| 09 | Advanced Patterns | Composition, mixins, dataclasses, design patterns |

### 🚀 Next Steps

- Build a **real project** using these patterns (e.g., a game, CLI tool, or web scraper)
- Explore **Python's standard library** for more OOP examples (`collections`, `pathlib`, `enum`)
- Learn about **type protocols** (`typing.Protocol`) for structural subtyping
- Study **SOLID principles** for writing maintainable OOP code
- Explore **metaclasses** for advanced class customization